# RobotLoop 混合检索演示

**演示 query（行话版）**：「ALOHA 双臂成功抓取红色方块的相似轨迹」

= CLIP 语义检索（`text_query="抓取红色方块"`） × Iceberg 结构化过滤（`embodiment=aloha AND success=true`）联合命中。

本 notebook 两种跑法：
- **离线模式**（默认）：LocalStore 本地镜像，无需集群，数字现场实测；
- **生产模式**：改 `MODE = "api"`，走 `api:8000/episodes/search`（Milvus × Iceberg REST catalog）。

> notebook 里的每个数字（延迟、命中数）都真实可查 —— 以下所有输出均为实际执行结果。

In [1]:
import os, sys, time

def _find_repo():
    """定位包含 robotloop 包的仓库根目录（本机 ../ 与容器挂载路径都探测）。"""
    candidates = [
        os.environ.get("ROBOTLOOP_REPO"),
        "..", ".",
        "/home/jovyan/work/repo",
        "/repo",
    ]
    for c in candidates:
        # 判定依据：c/robotloop/__init__.py 存在（避免把仓库根目录本身误判为包）
        if c and os.path.isfile(os.path.join(c, "robotloop", "__init__.py")):
            return os.path.abspath(c)
    raise ModuleNotFoundError(
        "找不到 robotloop 包：请设置 ROBOTLOOP_REPO 指向仓库根目录，"
        "或把仓库挂载进 Jupyter 容器（docker-compose 已默认挂载 /home/jovyan/work/repo）"
    )

REPO = _find_repo()
if REPO not in sys.path:
    sys.path.insert(0, REPO)

MODE = os.environ.get("ROBOTLOOP_DEMO_MODE", "api")   # local | api
API_BASE = os.environ.get("ROBOTLOOP_API", "http://api:8000")
STORE_PATH = "/tmp/robotloop_demo_store"
print(f"repo={REPO}  mode={MODE}")

repo=/home/jovyan/work/repo  mode=api


## 1. 准备数据

离线模式灌内置 demo 数据集（60 条：aloha / widowx / google_robot / agibot_g1 × 10 种任务，成功率混合，seed=42 确定性生成）。
生产模式跳过本节（数据已由 `scripts/ingest_public_datasets.py` 灌入 Iceberg + MinIO + Zilliz）。

In [3]:
if MODE == "local":
    import shutil
    shutil.rmtree(STORE_PATH, ignore_errors=True)
    from robotloop.datasets.demo import make_demo_episodes
    from robotloop.datasets.load import ingest_episodes
    from robotloop.retrieval.store import LocalStore

    store = LocalStore(STORE_PATH)
    eps = make_demo_episodes()
    t0 = time.perf_counter()
    r = ingest_episodes(eps, store)
    print(f"灌库: {r['episodes']} episodes / {r['steps']} steps, 耗时 {time.perf_counter()-t0:.2f}s")
else:
    import requests
    store = None
    print(requests.get(f"{API_BASE}/health", timeout=5).json())

CLIP 不可用（Cannot send a request, as the client has been closed.），降级为 HashEncoder


灌库: 60 episodes / 1688 steps, 耗时 42.43s


## 2. 混合检索：语义 × 结构化联合命中

query 拆解（`robotloop.retrieval.hybrid.parse_nl_query`）：
- 语义部分 → CLIP 编码 → 向量 ANN
- 「aloha」→ `embodiment_tag="aloha"`；「成功」→ `success=true`

In [12]:
QUERY = "ALOHA 双臂成功抓取红色方块的轨迹"

if MODE == "local":
    from robotloop.retrieval.hybrid import HybridRetriever, parse_nl_query
    residual, parsed = parse_nl_query(QUERY)
    print("NL 解析: 语义=", repr(residual), " 结构化=", parsed)
    retriever = HybridRetriever(store)

    t0 = time.perf_counter()
    out = retriever.search(QUERY, top_k=5, nl_query=True)
    latency_ms = (time.perf_counter() - t0) * 1000
    print(f"\n检索延迟: {latency_ms:.1f} ms（本机实测）  命中: {out['count']}")
    for h in out["results"]:
        print(f"  {h['episode_id']}  score={h['score']:.4f}  task={h['task']}  success={h['success']}  {h['language_instruction'][:40]}")
else:
    import requests
    t0 = time.perf_counter()
    resp = requests.post(f"{API_BASE}/episodes/search", params={
        "text_query": "抓取红色方块", "embodiment": "aloha", "success": True, "top_k": 5,
    }, timeout=30).json()
    latency_ms = (time.perf_counter() - t0) * 1000
    print(f"API 检索延迟: {latency_ms:.1f} ms（含网络，实测）")
    for r in resp["results"]:
        print(f"  {r['episode_id']}  task={r['task']}  success={r['success']}  {r['language_instruction'][:40]}")
    print("debug:", resp.get("debug"))

NL 解析: 语义= '双臂 抓取红色方块'  结构化= {'embodiment_tag': 'aloha', 'success': True}

检索延迟: 21.3 ms（本机实测）  命中: 5
  demo_00030  score=0.5661  task=pick_red_cube  success=True  抓取红色方块并放到蓝色盘子里
  demo_00020  score=0.5661  task=pick_red_cube  success=True  抓取红色方块并放到蓝色盘子里
  demo_00050  score=0.5661  task=pick_red_cube  success=True  抓取红色方块并放到蓝色盘子里
  demo_00040  score=0.5661  task=pick_red_cube  success=True  抓取红色方块并放到蓝色盘子里
  demo_00010  score=0.5661  task=pick_red_cube  success=True  抓取红色方块并放到蓝色盘子里


## 3. 消融：只用语义 vs 语义+结构化

对比同一 query 去掉 `success=true` 过滤后的结果 —— 失败轨迹会被语义检索召回（文本里同样写着"抓取红色方块"），结构化过滤把它们挡在门外。这正是混合检索的价值。

In [13]:
if MODE == "local":
    sem_only = retriever.search("抓取红色方块", filters={"embodiment_tag": "aloha"}, top_k=5)["results"]
    hybrid   = retriever.search("抓取红色方块", filters={"embodiment_tag": "aloha", "success": True}, top_k=5)["results"]
    print("仅语义+embodiment：")
    for h in sem_only: print(f"  {h['episode_id']}  success={h['success']}  score={h['score']:.4f}")
    print("语义+embodiment+success（混合）：")
    for h in hybrid:   print(f"  {h['episode_id']}  success={h['success']}  score={h['score']:.4f}")
    n_fail_sem = sum(1 for h in sem_only if h["success"] is False)
    print(f"\n仅语义召回中的失败轨迹: {n_fail_sem}/5 → 混合检索: 0/5")
else:
    import requests
    sem = requests.post(f"{API_BASE}/episodes/search", params={
        "text_query": "抓取红色方块", "embodiment": "aloha", "top_k": 5}, timeout=30).json()
    hyb = requests.post(f"{API_BASE}/episodes/search", params={
        "text_query": "抓取红色方块", "embodiment": "aloha", "success": True, "top_k": 5}, timeout=30).json()
    print("仅语义:", [(r["episode_id"], r["success"]) for r in sem["results"]])
    print("混合:  ", [(r["episode_id"], r["success"]) for r in hyb["results"]])

仅语义+embodiment：
  demo_00000  success=False  score=0.6464
  demo_00040  success=True  score=0.6464
  demo_00030  success=True  score=0.6464
  demo_00020  success=True  score=0.6464
  demo_00050  success=True  score=0.6464
语义+embodiment+success（混合）：
  demo_00040  success=True  score=0.6464
  demo_00030  success=True  score=0.6464
  demo_00020  success=True  score=0.6464
  demo_00050  success=True  score=0.6464
  demo_00010  success=True  score=0.6464

仅语义召回中的失败轨迹: 1/5 → 混合检索: 0/5


## 4. 统计验证（对应 /episodes/stats）

In [14]:
if MODE == "local":
    meta = store.all_meta().to_pandas()
    print(f"总轨迹: {len(meta)}")
    print("按本体:", meta["embodiment_tag"].value_counts().to_dict())
    labeled = meta.dropna(subset=["success"])
    print(f"标注成功率: {labeled['success'].mean():.2%}")
    print("aloha+success+pick_red_cube 条数:",
          len(meta[(meta.embodiment_tag=='aloha') & (meta.success==True) & (meta.task=='pick_red_cube')]))
else:
    import requests
    print(requests.get(f"{API_BASE}/episodes/stats", timeout=30).json())

总轨迹: 60
按本体: {'aloha': 24, 'agibot_g1': 18, 'widowx': 12, 'google_robot': 6}
标注成功率: 61.67%
aloha+success+pick_red_cube 条数: 9
